# RockWatch: Multi-Agent Geotechnical Early Warning Platform
### End-to-End Multi-Agent Execution Trace & Verification Notebook

This runnable notebook demonstrates the 8-agent blackboard-controller architecture executing the core operational scenarios:
1. **Scenario 1: Graceful Sensor Degradation (Primary Radar Offline)**
2. **Scenario 2: Un-alarmed Missed Detachment & Instant Caution Elevation**
3. **Scenario 3: Asymmetric Authority, Auto-Expiring Proposals & Model Validation Gate**
4. **Audit Trail Verification & Head Hash Anchoring**

In [ ]:
import json
import time
from blackboard import MineBlackboard, FORECASTER_BASELINE_SENSITIVITY
from controller import RockWatchController

# Initialize Controller & Shared Blackboard
blackboard = MineBlackboard(log_path="logs/run_log.jsonl")
controller = RockWatchController(blackboard=blackboard)
print(f"[*] RockWatch Multi-Agent Controller initialized.")
print(f"[*] Initial Ledger Head Hash: {blackboard.get_head_hash()}")

## 1. Scenario 1: Graceful Degradation (Primary Radar Offline)
A pit face loses primary radar telemetry. The system must not crash; instead, it falls back to secondary sensors (crack extensometer + weather), raises caution by one tier (Green -> Yellow per TARP-SOP-05), and widens its uncertainty interval.

In [ ]:
raw_frame_scenario_1 = {
    "radar_velocity_mm_hr": None,               # Primary radar offline
    "extensometer_velocity_mm_hr": 0.4,          # Secondary extensometer active
    "rainfall_3h_mm": 2.0,
    "vibration_ppv_mms": 1.0,
}

result_s1 = controller.run_telemetry_cycle("ZONE_NORTH_WALL", raw_frame_scenario_1)

print(f"=== MULTI-AGENT EXECUTION TRACE (SCENARIO 1) ===")
for step in result_s1["trace"]:
    print(f"Step {step['step']}: [{step['agent']}]")
    print(f"   Output: {json.dumps(step['output'], indent=2)[:200]}...\n")

print(f"Final Effective Alert Level: {result_s1['effective_level']}")
print(f"Alert Package Directive: {result_s1['final_alert_package']['advisory_directive']}")
print(f"Current Head Hash: {result_s1['head_hash']}")

## 2. Scenario 2: Unalarmed Missed Detachment & Instant Caution
LiDAR scan differencing detects an unalarmed 15.0 m3 detachment. The system triggers instant caution elevation, queues the event into the pending golden benchmark pool, and generates a diagnostic root-cause proposal for geotech sign-off.

In [ ]:
raw_frame_s2 = {"radar_velocity_mm_hr": 0.3, "rainfall_3h_mm": 0.0}
context_s2 = {
    "baseline_scan_volume_m3": 300.0,
    "current_scan_volume_m3": 285.0,  # 15 m3 detached
    "recent_alerts": [],              # No prior alerts = MISSED EVENT
}

result_s2 = controller.run_telemetry_cycle("ZONE_RAMP_3", raw_frame_s2, context=context_s2)

# Governance generates root-cause diagnostic
gov_res = controller.governance_agent.run(blackboard, {"zone_id": "ZONE_RAMP_3", "timestamp": time.time()})

print("=== MISSED EVENT DETECTION & GOVERNANCE REPORT ===")
print(f"Unalarmed Miss Detected: True")
print(f"Volume Detached: 15.0 m3")
print(f"Diagnostic Root Cause: {gov_res.get('root_cause_diagnosis', {}).get('proposed_root_cause')}")
print(f"Recommended Action: {gov_res.get('root_cause_diagnosis', {}).get('recommended_action')}")
print(f"Golden Pool Pending Events: {len(blackboard.read('golden_pool').get('pending_misses', []))}")

## 3. Scenario 3: Asymmetric Authority, Auto-Expiry & Model Gate
Demonstrates the Asymmetric Safety Principle:
- An operator's request to desensitize is rejected.
- A Geotechnical Engineer's authorization is approved.
- Simulated clock progression triggers code-level fail-closed expiry.
- Model Quality Gate rejects a retrained candidate model that degrades rockfall safety recall.

In [ ]:
# Step 1: Operator attempts desensitization (Rejected)
op_proposal = controller.submit_desensitization_proposal(
    zone_id="ZONE_SOUTH_CREST",
    operator_id="OP_ALICE",
    reason="Blast dust falsely triggered camera change",
    approver_role="OPERATOR",
    duration_hours=4.0
)
print(f"Operator Proposal Status: {op_proposal['proposal_result']['status']} (Approved: {op_proposal['proposal_result']['is_approved']})")

# Step 2: Geotechnical Engineer authorizes proposal (Approved)
now = time.time()
geo_proposal = controller.submit_desensitization_proposal(
    zone_id="ZONE_SOUTH_CREST",
    operator_id="OP_ALICE",
    reason="Blast dust verified by physical crest inspection",
    approver_role="GEOTECH_ENGINEER",
    duration_hours=4.0,
    requested_sensitivity=0.80
)
print(f"Geotech Proposal Status: {geo_proposal['proposal_result']['status']} (Approved: {geo_proposal['proposal_result']['is_approved']})")
print(f"Active Sensitivity: {blackboard.get_effective_forecaster_sensitivity('ZONE_SOUTH_CREST', now)}")

# Step 3: Fast-forward time past 4 hours to verify auto-expiry
future_time = now + (5.0 * 3600.0)
expired_sensitivity = blackboard.get_effective_forecaster_sensitivity('ZONE_SOUTH_CREST', future_time)
print(f"Sensitivity after 5 hours: {expired_sensitivity} (Reverted to Baseline!)")

# Step 4: Model Quality Gate rejects candidate model dropping recall
champion_model = {"recall": 0.96, "f1_score": 0.91, "precision": 0.86}
challenger_model = {"recall": 0.92, "f1_score": 0.92, "precision": 0.92}  # Drop of 4% recall!

gate_eval = controller.evaluate_challenger_model(champion_model, challenger_model)
print(f"\n=== MODEL BENCHMARK GATE VERDICT ===")
print(f"Gate Status: {gate_eval['gate_result']['status']}")
print(f"Promoted to Production: {gate_eval['gate_result']['promoted']}")
print(f"Verdict: {gate_eval['gate_result']['verdict_summary']}")

## 4. Audit Trail Verification & Head Hash Anchoring
Validates the complete append-only hash chain in `logs/run_log.jsonl` and prints the final head hash anchor.

In [ ]:
import hashlib

entries = []
with open("logs/run_log.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            entries.append(json.loads(line.strip()))

# Validate chain integrity
is_valid = True
for i in range(1, len(entries)):
    curr = entries[i].copy()
    expected_hash = curr.pop("hash")
    canonical_str = json.dumps(curr, sort_keys=True)
    recomputed = hashlib.sha256(canonical_str.encode("utf-8")).hexdigest()
    if recomputed != expected_hash or curr["prev_hash"] != entries[i-1]["hash"]:
        is_valid = False
        break

print(f"[*] Total Audit Ledger Entries: {len(entries)}")
print(f"[*] Hash-Chain Cryptographic Integrity Verified: {is_valid}")
print(f"[*] Head Hash Anchor: {blackboard.get_head_hash()}")